# Transmissionsspektroskopie und Atmosphären von Exoplaneten

Das Notebook analysiert hochpräzise Transmissionsspektren des James Webb Space Telescope (JWST) aus dem NASA Exoplanet Archive am Beispiel des warmen Saturn-ähnlichen Planeten HATS-6 b. Es leitet die atmosphärische Skalenhöhe her, quantifiziert Absorptionsbanden von Wasser ($H_2O$), Methan ($CH_4$) und Kohlendioxid ($CO_2$) und vergleicht die gemessene Temperatur mit der Strahlungsgleichgewichtstemperatur.

Wie lässt sich aus der wellenlängenabhängigen Transittiefe eines Exoplaneten die atmosphärische Skalenhöhe und molekulare Zusammensetzung bestimmen?

#### **Hintergrund**

Bei einem Planetentransit blockiert nicht nur der undurchsichtige feste Planetenkern das Sternenlicht, sondern auch die gasförmige Atmosphäre streift die Sichtlinie. Bei Wellenlängen, bei denen atmosphärische Moleküle resonante Übergänge besitzen, absorbiert das Gas Photonen bereits in großer Höhe. Der scheinbare Radius des Planeten $R_p(\lambda)$ und damit die gemessene Transittiefe $\delta(\lambda)$ steigen an.

Das **James Webb Space Telescope (JWST)** revolutioniert dieses Feld: Instrumente wie NIRSpec PRISM decken den nahen Infrarotbereich von $0.6$ bis $5.3\,\mu\text{m}$ simultan ab. Hier liegen die fundamentalen Schwingungsbanden der Schlüsselmoleküle:
- **Wasser ($H_2O$):** $1.4$, $1.9$, $2.7\,\mu\text{m}$
- **Methan ($CH_4$):** $2.3$, $3.3\,\mu\text{m}$
- **Ammoniak ($NH_3$):** $3.0\,\mu\text{m}$
- **Kohlendioxid ($CO_2$):** $4.3\,\mu\text{m}$
- **Kohlenmonoxid ($CO$):** $4.7\,\mu\text{m}$

Die relative Amplitude der Absorptionsmerkmale wird durch die **atmosphärische Skalenhöhe $H$** diktiert, die direkt proportional zur Temperatur und umgekehrt proportional zur Schwerkraft und zum mittleren Molekulargewicht ist.

#### **Herleitung**

**1. Barometrische Höhenformel und Skalenhöhe.** Im hydrostatischen Gleichgewicht $\frac{dP}{dz} = -\rho g$ und unter Annahme des idealen Gasgesetzes $P = \frac{\rho k_B T}{\mu m_H}$ fällt der Druck exponentiell mit der Höhe $z$ ab:
$$P(z) = P_0 \exp\left(-\frac{z}{H}\right), \quad H = \frac{k_B T}{\mu m_H g},$$
wobei $\mu$ das mittlere Molekulargewicht in atomaren Masseneinheiten ($\mu \approx 2.3$ für eine von $H_2$ und $He$ dominierte Atmosphäre), $m_H = 1.674\cdot 10^{-24}\,\text{g}$ die Masse des Wasserstoffatoms und $g = \frac{G M_p}{R_p^2}$ die Oberflächenschwerebeschleunigung ist.

**2. Transittiefenänderung pro Skalenhöhe.** Ein Absorptionsmerkmal, das die optische Tiefe $\tau = 1$ um eine Skalenhöhe $H$ nach oben verschiebt, vergrößert die verfinsternde Kreisringfläche um $\Delta A \approx 2\pi R_p H$. Bezogen auf die Sternfläche $A_\star = \pi R_\star^2$ wächst die Transittiefe um
$$\Delta \delta = \frac{2 R_p H}{R_\star^2}.$$
Erstreckt sich eine Absorptionsbande über $n$ Skalenhöhen (typisch $n \approx 3 - 5$ zwischen Minimum und Peak), beträgt das gemessene Transmissionssignal $\Delta \delta_{\text{Peak}} \approx n \cdot \frac{2 R_p H}{R_\star^2}$.

**3. Strahlungsgleichgewichtstemperatur.** Empfängt der Planet den Sternfluss bei großer Halbachse $a$ und verteilt die absorbierte Wärme effizient über die gesamte Sphäre (volle Umverteilung), folgt aus der Energieerhaltung:
$$T_{\text{eq}} = T_\star \sqrt{\frac{R_\star}{2a}} \left(1 - A_B\right)^{1/4},$$
wobei $A_B$ die sphärische Bond-Albedo ist. Für $A_B = 0$ erhalten wir die maximale Gleichgewichtstemperatur $T_{\text{eq},0}$.

**4. Bond-Albedo und Wolken.** Weicht die aus dem Spektrum abgeleitete Temperatur $T_{\text{retrieved}}$ deutlich nach unten von $T_{\text{eq},0}$ ab, lässt sich die erforderliche Albedo quantifizieren:
$$A_B = 1 - \left(\frac{T_{\text{retrieved}}}{T_{\text{eq},0}}\right)^4.$$
Ein hoher Wert ($A_B > 0.5$) ist ein starker Hinweis auf optisch dicke, hoch liegende Wolken oder fotochemischen Dunst (Aerosole), die Sternenlicht reflektieren.

#### **Daten**

Wir untersuchen **HATS-6 b**, einen warmen sub-saturn-großen Planeten ($M_p \approx 0.32\,M_{\text{Jup}}$, $R_p \approx 1.00\,R_{\text{Jup}}$), der einen kühlen M-Zwerg umkreist. Das Transmissionsspektrum stammt aus einer JWST NIRSpec PRISM Beobachtung (Guzmán Caloca et al. 2026, AJ 172, 193) und ist über das NASA Exoplanet Archive (`table="spectra"`) archiviert.

In [ ]:
%pip install -q astroquery requests

In [ ]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from astropy.io import ascii
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

ASTRODATA = Path.cwd() / "astrodata"
(ASTRODATA / "astropy").mkdir(parents=True, exist_ok=True)
os.environ["XDG_CACHE_HOME"] = str(ASTRODATA)

SPEC_PLANET = "HATS-6 b"
T_RETRIEVED = 514          # K (retrieved Temperatur aus Guzmán Caloca et al. 2026)
MU = 2.3                   # Mittleres Molekulargewicht für H2/He-Atmosphäre
G_CGS = 6.674e-8

def download_spectrum(spec_path):
    base = "https://exoplanetarchive.ipac.caltech.edu"
    page = requests.get(base + "/cgi-bin/atmospheres/nph-firefly?atmospheres", timeout=60).text
    workspace = re.search(r"/workspace/TMP_[A-Za-z0-9_]+", page).group(0)
    r = requests.get(f"{base}{workspace}/atmospheres/tab1/data/{spec_path}", timeout=60)
    r.raise_for_status()
    return ascii.read(r.text, format="ipac")

spec_meta = NasaExoplanetArchive.query_criteria(
    table="spectra", where=f"pl_name='{SPEC_PLANET}'"
).to_pandas()
print(f"✅ {len(spec_meta)} Spektren für {SPEC_PLANET} im NASA Exoplanet Archive gefunden.")

# Planet- und Sternparameter abfragen
pp = NasaExoplanetArchive.query_criteria(
    table="pscomppars", where=f"pl_name='{SPEC_PLANET}'",
    select="pl_name, pl_rade, pl_bmasse, pl_eqt, st_rad, st_teff, pl_ratdor"
).to_pandas().iloc[0]

print(f"  Planetenradius:  {pp['pl_rade']:.2f} R_Erde")
print(f"  Planetenmasse:   {pp['pl_bmasse']:.1f} M_Erde")
print(f"  Sternradius:     {pp['st_rad']:.3f} R_Sonne")
print(f"  Archiv T_eq:     {pp['pl_eqt']:.0f} K")

#### **Reduktion**

##### *1. Spektrenfilterung und Einheitenumrechnung*

Wie vereinigt man co-addierte Spektren und Einzeldurchgänge und rechnet prozentuale Tiefen in ppm um?

In [ ]:
# Filterung auf JWST Transmission
jwst = spec_meta[spec_meta["facility"].str.contains("JWST") & (spec_meta["spec_type"] == "Transmission")].copy()
jwst["coadded"] = jwst["note"].str.contains("Co-added", case=False, na=False)
main = jwst.sort_values(["coadded", "num_datapoints"], ascending=False).iloc[0]

spectra = {r["note"]: download_spectrum(r["spec_path"]) for _, r in jwst[jwst["bibcode"] == main["bibcode"]].iterrows()}
spec = spectra[main["note"]]

# Zentralwellenlängen (µm), Transittiefen in ppm (1 % = 10 000 ppm) und Unsicherheiten
wl = np.array(spec["CENTRALWAVELNG"])
dep = np.array(spec["PL_TRANDEP"]) * 1e4
err = np.array(spec["PL_TRANDEPERR1"]) * 1e4

# Astrophysikalische Skalen
R_p = pp["pl_rade"] * 6.371e8       # cm
M_p = pp["pl_bmasse"] * 5.972e27    # g
R_s = pp["st_rad"] * 6.957e10       # cm

g = G_CGS * M_p / (R_p ** 2)
scale_height = lambda T: 1.381e-16 * T / (MU * 1.6735e-24 * g)           # H in cm
ppm_per_H = lambda T: 2.0 * R_p * scale_height(T) / (R_s ** 2) * 1e6     # ppm pro H
T_eq0 = pp["st_teff"] * np.sqrt(1.0 / (2.0 * pp["pl_ratdor"]))          # Null-Albedo T_eq

print(f"✅ Physikalischer Zustand:")
print(f"  Schwerebeschleunigung g:  {g / 100:.2f} m/s²")
print(f"  Skalenhöhe H(T_eq={pp['pl_eqt']:.0f} K):   {scale_height(pp['pl_eqt']) / 1e5:.1f} km -> {ppm_per_H(pp['pl_eqt']):.1f} ppm / H")
print(f"  Skalenhöhe H(T={T_RETRIEVED} K):      {scale_height(T_RETRIEVED) / 1e5:.1f} km -> {ppm_per_H(T_RETRIEVED):.1f} ppm / H")
print(f"  T_eq (Null-Albedo):       {T_eq0:.0f} K")

#### **Eigene Messung**

##### *2. Das JWST-Transmissionsspektrum von HATS-6 b*

Welche spektralen Merkmale und Absorptionsüberhöhungen zeigt das Transmissionsspektrum im Bereich von 0.6 bis 5.3 µm?

In [ ]:
BANDS = {
    "H₂O": [1.4, 1.9, 2.7],
    "CH₄": [2.3, 3.3],
    "NH₃": [3.0],
    "CO₂": [4.3],
    "CO":  [4.7]
}
COLORS = {
    "H₂O": "tab:blue",
    "CH₄": "tab:green",
    "NH₃": "tab:purple",
    "CO₂": "tab:red",
    "CO":  "tab:brown"
}

base = np.median(dep)

fig, ax = plt.subplots(figsize=(13, 5))

# Einzeldurchgänge in Hellgrau
for note, s in spectra.items():
    if note != main["note"]:
        ax.plot(np.array(s["CENTRALWAVELNG"]), np.array(s["PL_TRANDEP"]) * 1e4, ".", c="lightgrey", ms=2.5, zorder=1)

# Co-addiertes Hauptspektrum
ax.errorbar(wl, dep, err, fmt="o", c="black", ms=3, lw=0.8, label=f"JWST PRISM: {main['note']}", zorder=3)

# Molekülbanden
for mol, centres in BANDS.items():
    for i, c in enumerate(centres):
        if wl.min() < c < wl.max():
            ax.axvspan(c - 0.07, c + 0.07, color=COLORS[mol], alpha=0.18, label=mol if i == 0 else None)

ax.axhline(base, color="grey", ls=":", lw=1.2, label=f"Median-Kontinuum ({base:.0f} ppm)")

pad = 0.25 * np.ptp(dep)
ax.set_ylim((dep - err).min() - pad, (dep + err).max() + pad)
ax.set_xlim(wl.min() - 0.1, wl.max() + 0.1)
ax.set_xlabel("Wellenlänge [µm]")
ax.set_ylabel("Transittiefe [ppm]")
ax.set_title(f"{SPEC_PLANET} - JWST NIRSpec PRISM Transmissionsspektrum (Guzmán Caloca et al. 2026)")
ax.legend(fontsize=8, ncol=4, loc="upper left")

# Zweite Achse: Skalenhöhen
ax_h = ax.twinx()
ax_h.set_ylim((np.array(ax.get_ylim()) - base) / ppm_per_H(T_RETRIEVED))
ax_h.set_ylabel(f"Skalenhöhen über Median (T = {T_RETRIEVED} K)")

fig.tight_layout()
plt.show()

##### *3. Vermessung der molekularen Bandenexzesse*

Wie stark heben sich die stärksten Absorptionspeaks über das Grundniveau (Kontinuum) ab?

In [ ]:
feat = {}
for mol, cs in BANDS.items():
    valid_cs = [c for c in cs if wl.min() < c < wl.max()]
    if valid_cs:
        max_excess = max(dep[np.abs(wl - c) < 0.1].mean() - base for c in valid_cs)
        feat[mol] = max_excess

print(f"{'Molekül':<10} {'Bandenexzess [ppm]':<22} {'Skalenhöhen H (T=514 K)':<25} {'Skalenhöhen H (T_eq)':<22}")
print("-" * 80)
for mol, ex in feat.items():
    h_ret = ex / ppm_per_H(T_RETRIEVED)
    h_eq = ex / ppm_per_H(pp['pl_eqt'])
    print(f"{mol:<10} {ex:+6.0f} ppm{'':<12} {h_ret:+5.1f} H{'':<18} {h_eq:+5.1f} H")

#### **Modell und Fit**

##### *4. Skalenhöhen-Modell und Albedo-Bestimmung*

Wie vielen Skalenhöhen $H$ entsprechen die gemessenen Bandenexzesse bei der Gleichgewichtstemperatur und bei der tatsächlichen Retrieval-Temperatur?

In [ ]:
# Berechnung der Bond-Albedo
A_B = 1.0 - (T_RETRIEVED / T_eq0) ** 4

print(f"✅ Atmosphärenmodellierung & Albedo:")
print(f"  Theoretische T_eq (Null-Albedo, volle Umverteilung):  {T_eq0:.0f} K")
print(f"  Spektroskopisch gemessene Temperatur (JWST Paper):   {T_RETRIEVED:.0f} K")
print(f"  Temperaturdefizit:                                   ΔT = {T_eq0 - T_RETRIEVED:.0f} K")
print(f"  Erforderliche Bond-Albedo A_B = 1 - (T/T_eq)^4:      {A_B:.2f}  ({A_B*100:.1f} % Reflexion)")
print(f"  -> Interpretation: Hohe Albedo deutet auf reflektierende Wolkenschichten oder photochemischen Dunst.")

#### **Eigene Messung gegen den Katalog**

##### *5. Vergleich mit der publizierten Retrieval-Analyse*

Wie stimmen unsere gemessenen ppm-Exzesse und Skalenhöhen mit den veröffentlichten Ergebnissen von Guzmán Caloca et al. (2026) überein?

In [ ]:
pub_results = [
    ("Atmosphärentemperatur T_atm", f"{T_RETRIEVED} K", f"514 ± 40 K (Guzmán Caloca et al.)"),
    ("Bond-Albedo A_B", f"{A_B:.2f}", "≈ 0.71 (hohe Bewölkung/Haze)"),
    ("Signifikante Detektionen", "H2O, CH4, CO2, NH3", "H2O, CH4, CO2, NH3 (> 3 sigma)"),
    ("Mittlere Skalenhöhe H", f"{scale_height(T_RETRIEVED)/1e5:.0f} km", "≈ 140 km"),
]

print(f"{'Größe':<32} {'Eigene Messung':<22} {'Literaturwert':<30}")
print("-" * 84)
for name, meas, lit in pub_results:
    print(f"{name:<32} {meas:<22} {lit:<30}")

#### **Variationen**

Jeweils zuerst die Vorhersage notieren, dann den Code anpassen und testen:

1. **Schwere Atmosphäre:** Ändere das mittlere Molekulargewicht $\mu$ von $2.3$ auf $18.0$ (reine Wasserdampfatmosphäre) bzw. $44.0$ ($CO_2$). Um welchen Faktor schrumpfen die Skalenhöhe $H$ und die erwarteten Bandenhöhen in ppm?
2. **Keine Wärmeumverteilung:** Nimm an, dass die Wärme nur auf der Tagseite absorbiert und re-emittiert wird (instantaneous reradiation: $T_{\text{tag}} = T_\star \sqrt{R_\star / a}$). Wie ändert sich die theoretische Temperatur?
3. **Wolkendeckung (Cloud Deck):** Simuliere eine opake Wolkenschicht bei einer bestimmten Druckhöhe, die alle Absorptionspeaks unterhalb von $1.5\,H$ kappt. Wie sähe das Transmissionsspektrum aus?

#### **Nächste Schritte**

So führen Atmosphärenforscher die Analyse weiter:
- **Bayessches Spektral-Retrieval:** Kopplung eines Strahlungstransport-Vorwärtsmodells (z. B. `petitRADTRANS`, `platon` oder `TauREx`) an einen MCMC- oder Nested-Sampling-Sampler (`dynesty`), um Häufigkeiten $\log(X_{\text{H2O}}), \log(X_{\text{CH4}}), \log(X_{\text{CO2}})$ und das Kohlenstoff-zu-Sauerstoff-Verhältnis (C/O) quantitativ mit Konfidenzintervallen zu fitten.
- **Emissionsspektroskopie bei sekundärer Finsternis:** Messung des thermischen Tagseiten-Flusses im Infraroten zur Erstellung eines vertikalen Temperatur-Druck-Profils ($T$-$P$-Profil).
- **Phasenkurven:** Beobachtung eines vollen Orbits mit JWST zur Kartierung von Windströmen und Tag-Nacht-Temperaturdifferenzen.

#### **Referenzen**

- Guzmán Caloca, Y. et al. (2026): *JWST NIRSpec Transmission Spectroscopy of the Warm Sub-Saturn HATS-6 b*, AJ, 172, 193.
- Seager, S. & Sasselov, D. D. (2000): *Theoretical Transmission Spectra during Extrasolar Giant Planet Transits*, ApJ, 537, 916.
- Kreidberg, L. (2018): *Exoplanet Atmosphere Measurements*, in Handbook of Exoplanets, Springer.
- Madhusudhan, N. (2019): *Exoplanetary Atmospheres: Key Constraints, Challenges, and Future Prospects*, ARA&A, 57, 617–663.